# 06 - Research Experiment Harness (Intro)

A minimal grid-search harness that writes **JSONL logs**.


In [1]:
import os
import requests

# Resolve Ollama endpoint depending on environment
BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')
print('Using Ollama at:', BASE_URL)

# Diagnostic check
try:
    r = requests.get(BASE_URL, timeout=3)
    print('Server reachable. Status:', r.status_code)
    try:
        tags = requests.get(f"{BASE_URL}/api/tags", timeout=3)
        print('Tags endpoint status:', tags.status_code)
    except Exception as e:
        print('Tags endpoint check failed:', e)
except Exception as e:
    print('Server not reachable:', e)


Using Ollama at: http://host.docker.internal:11434
Server reachable. Status: 200
Tags endpoint status: 200


In [2]:
import os, json, time
import requests


In [3]:
BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')

def run(model: str, prompt: str, temperature: float):
    payload = {'model': model, 'prompt': prompt, 'temperature': temperature, 'stream': False}
    t0 = time.time()
    r = requests.post(f'{BASE_URL}/api/generate', json=payload, timeout=300)
    r.raise_for_status()
    t1 = time.time()
    out = r.json().get('response', '')
    return {
        'ts': time.time(),
        'model': model,
        'temperature': temperature,
        'prompt': prompt,
        'latency_s': t1 - t0,
        'output_chars': len(out)
    }



In [4]:
MODELS = ['llama3.2:3b']
TEMPS = [0.0, 0.2, 0.8]
PROMPTS = [
  'Explain RAG in a systems perspective.',
  'Give a short comparison: local inference vs cloud inference.',
  'Write pseudocode for SGD.'
]


In [5]:
rows = []
for m in MODELS:
  for t in TEMPS:
    for p in PROMPTS:
      rows.append(run(m, p, t))

os.makedirs('../experiments/logs', exist_ok=True)
path = '../experiments/logs/experiments.jsonl'
with open(path, 'w') as f:
  for r in rows:
    f.write(json.dumps(r) + '\n')

print('Saved', path)
rows[:2]


Saved ../experiments/logs/experiments.jsonl


[{'ts': 1772715847.6552055,
  'model': 'llama3.2:3b',
  'temperature': 0.0,
  'prompt': 'Explain RAG in a systems perspective.',
  'latency_s': 6.55647349357605,
  'output_chars': 3285},
 {'ts': 1772715850.1280305,
  'model': 'llama3.2:3b',
  'temperature': 0.0,
  'prompt': 'Give a short comparison: local inference vs cloud inference.',
  'latency_s': 2.4726359844207764,
  'output_chars': 1302}]